## $$\text{Procedures for the Grid Convergence Index (GCI)}$$

#### $\text{Step 1}$

We have to define a representative cell size $h$

$$
h = \left[\frac{1}{N} \sum_{i=1}^{N} \left(\Delta A_i \right) \right]^{1/2}
\tag{1}
$$

However, since the computational domain is the same for all three grids, we can use

$$
h \propto N^{-1/2}
\tag{2}
$$

Since

$$
A_\text{tot} = \sum_{i=1}^N \Delta A_i \implies h = \left(\frac{A_\text{tot}}{N} \right)^{1/2}
$$

and

$$
A_{\text{tot},1} = A_{\text{tot},2} = A_{\text{tot},3},
$$

we get

$$
r_{21} = \frac{h_2}{h_1}
= \frac{(A_{\text{tot}}/N_2)^{1/2}}{(A_{\text{tot}}/N_1)^{1/2}}
= \left(\frac{N_1}{N_2}\right)^{1/2}
\tag{3}
$$

Thus,

$$
h_2 \propto N_2^{-1/2}, \quad h_1 \propto N_1^{-1/2}
$$

Note that the refinement factor $r$ should be greater than 1.3.

#### $\text{Step 2}$

Next we have to create three *significant* different meshes and run the simulation to the key variable, for us $C_d$ (drag coefficient). Often we denote the key variable as $\phi$

#### $\text{Step 3}$

Let $h_1 < h_2 < h_3$ and calculate the apparent order p

$$
p = \frac{1}{\text{ln}(r_{21})}\ |\text{ln}(|\varepsilon_{32}/ \varepsilon_{21}|) + q(p)
\tag{4}
$$

Where 
$$
\varepsilon_{32} = \phi_3 - \phi_2,\quad \varepsilon_{21} = \phi_2 - \phi_1,\quad q(p) = \text{ln}\left(\frac{r_{21}^{p} - s}{r_{32}^{p} - s} \right),
\quad s = 1\cdot \text{sgn}(\varepsilon_{32}/\varepsilon_{21})
$$

Equation 4 can be solved using fixed-point iteration, with an initial guess to the first term. Also $\varepsilon_{32},\ \varepsilon_{21}$ very close to zero does not work. Which is an indication of oscillatory convergence. Then further calculations have to be performed.

#### $\text{Step 4}$

Calculate the extrapolated (ext) values from

$$
\phi_\text{ext}^{21} = \frac{r_{21}^p \phi_1 - \phi_2}{r_{21}^p - 1},\quad \phi_\text{ext}^{32} = \frac{r_{32}^p \phi_1 - \phi_2}{r_{32}^p - 1}
$$

#### $\text{Step 5}$

Calculate the error estimate, along with the apparent order p

$$
\text{Approximate relative error}\to e_a^{21} = \left|\frac{\phi_1 - \phi_2}{\phi_1} \right|,\quad
$$

$$
\text{Extrapolated relative error}\to e_{ext}^{21} = \left|\frac{\phi_\text{ext}^{21} - \phi_1}{\phi_\text{ext}^{21}} \right|
$$

Finally

$$
\text{GCI}_\text{fine}^{21} = \frac{1.25 \cdot e_a^{21}}{r_{21}^p - 1},\quad \text{GCI}_\text{fine}^{32} = \frac{1.25 \cdot e_a^{32}}{r_{32}^p - 1}
$$

In [ ]:
import pandas as pd
import numpy as np

# --- Load data ---
d3 = pd.read_csv("Data/GCI/coarse.dat", sep=r"\s+")
d4 = pd.read_csv("Data/GCI/medium.dat", sep=r"\s+")
d5 = pd.read_csv("Data/GCI/fine.dat", sep=r"\s+")

start = 100
stop  = 8000

y3 = d3["Cd"][start:stop].to_numpy()
y4 = d4["Cd"][start:stop].to_numpy()
y5 = d5["Cd"][start:stop].to_numpy()

def amplitude(y):
    return 0.5 * (np.max(y) - np.min(y))

# Quantity of interest: fine, medium, coarse
phi1 = amplitude(y5)
phi2 = amplitude(y4)
phi3 = amplitude(y3)

# --- Grid sizes ---
N1, N2, N3 = 32500, 13248, 5748   # fine, medium, coarse

dim = 2  # use 3 if this is a quasi-2D extruded 3D OpenFOAM mesh

h1 = N1 ** (-1.0 / dim)
h2 = N2 ** (-1.0 / dim)
h3 = N3 ** (-1.0 / dim)

r21 = h2 / h1
r32 = h3 / h2

if not (r21 > 1.0 and r32 > 1.0):
    raise ValueError("Refinement ratios must satisfy r21 > 1 and r32 > 1.")

# --- Solution differences (signed) ---
eps21 = phi2 - phi1
eps32 = phi3 - phi2

if np.isclose(eps21, 0.0) or np.isclose(eps32, 0.0):
    raise ValueError("One of the solution differences is too close to zero to compute p reliably.")

s = np.sign(eps32 / eps21)

# --- Apparent order p via fixed-point iteration ---
p = 2.0
tol = 1e-10
max_iter = 100

for _ in range(max_iter):
    denom_term = (r32**p - s)
    numer_term = (r21**p - s)

    if numer_term <= 0 or denom_term <= 0:
        raise ValueError("Invalid argument in logarithm during p iteration.")

    p_new = (
        np.log(abs(eps32 / eps21)) + np.log(numer_term / denom_term)
    ) / np.log(r21)

    p_new = abs(p_new)

    if abs(p_new - p) < tol:
        p = p_new
        break

    p = p_new
else:
    raise RuntimeError("Fixed-point iteration for p did not converge.")

# --- Richardson extrapolation ---
if np.isclose(r21**p - 1.0, 0.0):
    raise ValueError("Richardson extrapolation denominator is too small.")

phi_ext = (r21**p * phi1 - phi2)/(r21**p - 1)

# --- Relative errors ---
e21 = abs((phi1 - phi2) / phi1)
e32 = abs((phi2 - phi3) / phi2)

# --- GCI ---
Fs = 1.25

GCI21 = Fs * e21 / (r21**p - 1.0)
GCI32 = Fs * e32 / (r32**p - 1.0)

# --- Asymptotic range check ---
AR = GCI32 / (GCI21 * r21**p)

# --- Output ---
print(f"phi1 (fine)        = {phi1:.6f}")
print(f"phi2 (medium)      = {phi2:.6f}")
print(f"phi3 (coarse)      = {phi3:.6f}")
print()
print(f"h1                 = {h1:.6e}")
print(f"h2                 = {h2:.6e}")
print(f"h3                 = {h3:.6e}")
print()
print(f"r21                = {r21:.6f}")
print(f"r32                = {r32:.6f}")
print()
print(f"eps21              = {eps21:.6e}")
print(f"eps32              = {eps32:.6e}")
print(f"s                  = {s:.0f}")
print()
print(f"p (observed order) = {p:.6f}")
print(f"phi_ext            = {phi_ext:.6f}")
print()
print(f"GCI21              = {100*GCI21:.3f} %")
print(f"GCI32              = {100*GCI32:.3f} %")
print(f"Asymptotic ratio   = {AR:.6f}")

phi1 (fine)        = 0.212105
phi2 (medium)      = 0.205189
phi3 (coarse)      = 0.208377

h1                 = 5.547002e-03
h2                 = 8.688101e-03
h3                 = 1.318990e-02

r21                = 1.566270
r32                = 1.518157

eps21              = -6.915500e-03
eps32              = 3.188000e-03
s                  = -1

p (observed order) = 1.648858
phi_ext            = 0.218417

GCI21              = 3.720 %
GCI32              = 1.961 %
Asymptotic ratio   = 0.251519
